In [ ]:
# Package path adaptation only: keep all legacy scientific calculations unchanged.
from pathlib import Path as _PackagePath
import os as _package_os
_package_candidates = (_PackagePath.cwd(), _PackagePath.cwd().parent, _PackagePath.cwd() / "single_period")
_package_root = next((p for p in _package_candidates if (p / "notebooks").is_dir() and (p / "pinn_results").is_dir()), None)
if _package_root is None:
    raise RuntimeError("Start this notebook from the single_period package or one of its immediate subdirectories.")
_package_os.chdir(_package_root)
(_package_root / "generated" / "one-period-result-fig").mkdir(parents=True, exist_ok=True)
(_package_root / "generated" / "comparison_results").mkdir(parents=True, exist_ok=True)
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import cm

In [ ]:
# 参数设置
T = 10           # 终止时间
dt = 0.01           # 时间步长
N = int(T / dt)     # 时间步数量
n_simulations = 10000  # 模拟路径数量

# SDE参数
pi = 100             # pi常数
beta = 0.161         # 参数beta
alpha = 0.332        # 参数alpha
r = 0.05                # 无风险利率
K = 50.0                # 行权价


In [ ]:

# 初始化路径数组
A_paths = np.zeros((n_simulations, N+1))

# 初始条件
A_paths[:, 0] = 0.5 * pi  # 假设初始值 A_0 = 0.5 * pi

# 时间序列
time_grid = np.linspace(0, T, N+1)

# 模拟路径
for i in range(N):
    t = i * dt
    sqrt_zt = np.sqrt(beta * (T - t)**-alpha)
    # \Phi 和 \Phi^{-1} 的计算
    Phi_inv_A = stats.norm.ppf(A_paths[:, i] / pi)  # 计算 \Phi^{-1}(A/pi)
    Phi_prime = stats.norm.pdf(Phi_inv_A)           # 计算 \Phi'(Phi^{-1}(A/pi))

    # dA_t = pi * \Phi'(Phi^{-1}(A/pi)) * sqrt(z_t) * dW_t
    dW = np.sqrt(dt) * np.random.randn(n_simulations)
    dA = pi * Phi_prime * sqrt_zt * dW
    A_paths[:, i+1] = A_paths[:, i] + dA

# 分析结果：计算终点的分布
A_T = A_paths[:, -1]

# 绘制直方图和核密度估计
plt.figure(figsize=(10, 6))
plt.hist(A_T, bins=50, density=True, alpha=0.5, color='blue', label='Histogram of $A_T$')
sns.kdeplot(A_T, color='red', label='KDE of $A_T$')
plt.xlabel('$A_T$')
plt.ylabel('Density')
plt.title(f'Distribution of $A_T$ at T={T}')
plt.legend()
plt.show()


In [ ]:
# 计算期权到期支付
payoffs = np.maximum(A_paths[:, -1] - K, 0)
mc_price = np.exp(-r * T) * np.mean(payoffs)
print(f"蒙特卡洛模拟价格: {mc_price:.4f}")


In [ ]:

# 初始化路径数组
A_paths = np.zeros((n_simulations, N+1))
A_paths[:, 0] = 0.5 * pi  # 初始条件 A_0 = 0.5 * pi

# 时间序列
time_grid = np.linspace(0, T, N+1)

# 模拟路径
for i in range(N):
    t = i * dt
    sqrt_zt = np.sqrt(beta * (T - t)**-alpha)
    Phi_inv_A = stats.norm.ppf(A_paths[:, i] / pi)  # 计算 \Phi^{-1}(A/pi)
    Phi_prime = stats.norm.pdf(Phi_inv_A)           # 计算 \Phi'(Phi^{-1}(A/pi))

    # dA_t = pi * \Phi'(Phi^{-1}(A/pi)) * sqrt(z_t) * dW_t
    dW = np.sqrt(dt) * np.random.randn(n_simulations)
    dA = pi * Phi_prime * sqrt_zt * dW
    A_paths[:, i+1] = A_paths[:, i] + dA

In [ ]:
# 绘制时间演变的KDE图
plt.figure(figsize=(12, 8))

# 选取多个时间节点来绘制KDE
time_points = np.linspace(0, N, 10, dtype=int)  # 选取10个时间点
colors = cm.viridis(np.linspace(0, 1, len(time_points)))[::-1]  # 生成从深到浅的颜色渐变

for i, (t, color) in enumerate(zip(time_points, colors)):
    sns.kdeplot(A_paths[:, t], bw_adjust=0.5, color=color, label=f'Time = {time_grid[t]:.2f}')

plt.xlabel('A(t)')
plt.ylabel('Density')
plt.title('Time-evolving KDE of $A_t$ with Gradient Color')
plt.legend()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from matplotlib import cm

# 参数设置
# n_simulations = 10000  # 模拟路径数量
# N = 1000                # 时间步数
# T = 4.0                 # 总时间（与你的PINN模型一致）
# dt = T / N
# pi = 100.0              # 参数 pi（与PINN模型一致）
# beta = 0.5              # 参数 beta（与PINN模型一致）
# alpha = 0.8             # 参数 alpha（与PINN模型一致）


# 初始化路径数组
A_paths = np.zeros((n_simulations, N + 1))
A_paths[:, 0] = 0.5 * pi  # 初始值 A_0 = 0.5 * pi

# 时间网格
time_grid = np.linspace(0, T, N + 1)

# # 模拟路径
# for i in range(N):
#     t = i * dt
#     sqrt_zt = np.sqrt(beta * (T - t) ** (-alpha))
#     Phi_inv_A = stats.norm.ppf(A_paths[:, i] / pi)
#     Phi_prime = stats.norm.pdf(Phi_inv_A)
#     dW = np.sqrt(dt) * np.random.randn(n_simulations)
#     dA = pi * Phi_prime * sqrt_zt * dW
#     A_paths[:, i + 1] = A_paths[:, i] + dA

# 计算期权到期支付
payoffs = np.maximum(A_paths[:, -1] - K, 0)
mc_price = np.exp(-r * T) * np.mean(payoffs)
print(f"蒙特卡洛模拟价格: {mc_price:.4f}")

# # 获取PINN预测价格（假设A=0.5*pi, t=0）
# A_input = torch.tensor([[0.5 * pi]], device=device)  # 初始标的资产价格
# t_input = torch.tensor([[0.0]], device=device)       # 当前时间t=0
# pinn_price = model(A_input, t_input).item()
# print(f"PINN模型预测价格: {pinn_price:.4f}")

# # 计算误差
# error = abs(pinn_price - mc_price) / mc_price
# print(f"相对误差: {error:.2%}")

In [ ]:
A_paths